# Semaine 5, Session 2 : Arbres & Parcours (DFS/BFS)

## Ce que vous apprendrez
- Comprendre la structure des arbres : nœuds, arêtes, racine, feuilles
- Maîtriser les concepts d'arbre binaire et d'ABR (Arbre Binaire de Recherche)
- Implémenter les parcours d'arbres : infixe, préfixe, postfixe
- Comprendre BFS vs DFS et quand utiliser chacun

---

## Partie 1 : Fondements des arbres

### Terminologie des arbres

**Arbre** = Structure hiérarchique avec des nœuds reliés par des arêtes.

**Exemple visuel :**
```
        1        ← Racine (profondeur 0)
       / \
      2   3      ← Enfants de 1 (profondeur 1)
     / \
    4   5        ← Feuilles (profondeur 2, pas d'enfants)
```

**Termes clés :**
- **Racine** : Nœud du sommet (pas de parent)
- **Nœud** : Contient des données + des références vers les enfants
- **Feuille** : Nœud sans enfants
- **Profondeur** : Distance depuis la racine (racine = 0)
- **Hauteur** : Chemin le plus long du nœud à une feuille

**Arbre binaire :** Chaque nœud a au plus 2 enfants (gauche et droite)

In [12]:
# Nœud d'arbre utilisant un dictionnaire
# Chaque nœud est un dictionnaire avec les clés 'val', 'left' et 'right'
# Pas besoin de classes ! Juste des dictionnaires !

# Fonction utilitaire pour créer un nœud (optionnelle, mais rend le code plus propre)
def create_tree_node(val, left=None, right=None):
    """Créer un dictionnaire représentant un nœud d'arbre"""
    return {'val': val, 'left': left, 'right': right}

# Créer un arbre simple :
#       1
#      / \
#     2   3
#    / \
#   4   5

root = create_tree_node(1)
root['left'] = create_tree_node(2)
root['right'] = create_tree_node(3)
root['left']['left'] = create_tree_node(4)
root['left']['right'] = create_tree_node(5)

# Alternative : Créer directement avec des dictionnaires
# root = {'val': 1, 'left': {'val': 2, 'left': {'val': 4, 'left': None, 'right': None}, 
#                            'right': {'val': 5, 'left': None, 'right': None}}, 
#         'right': {'val': 3, 'left': None, 'right': None}}

### Arbre Binaire de Recherche (ABR)

**Propriété de l'ABR :**
- Le sous-arbre gauche contient des valeurs < nœud
- Le sous-arbre droit contient des valeurs > nœud
- Les deux sous-arbres sont aussi des ABR

**Qu'est-ce que cela signifie ?** Pour tout nœud, TOUTES les valeurs de son sous-arbre gauche sont plus petites, et TOUTES les valeurs de son sous-arbre droit sont plus grandes.

**Exemple - ABR valide :**
```
       4
      / \
     2   6    ← 2 < 4, 6 > 4 ✓
    / \ / \
   1  3 5  7  ← Toutes les valeurs à gauche < parent, à droite > parent ✓
```

**Exemple - ABR invalide :**
```
       4
      / \
     2   6
    / \
   1   5    ← PROBLÈME : 5 > 2 (OK), mais 5 > 4 (PAS OK !)
```

**Pourquoi est-ce invalide ?**
- Le nœud 5 est dans le sous-arbre GAUCHE du nœud 4
- Mais 5 > 4, donc 5 devrait être dans le sous-arbre DROIT du nœud 4 !
- Règle : TOUTES les valeurs du sous-arbre gauche doivent être < parent
- Comme 5 est dans le sous-arbre gauche mais 5 > 4, cela viole la règle de l'ABR ✗

**Avantage :** Permet une recherche efficace (O(log n) en moyenne) - on peut éliminer la moitié de l'arbre à chaque étape !
- Chercher 5 ? Commencer à 4, 5 > 4, aller à droite (éliminer la moitié gauche !)
- Chercher 1 ? Commencer à 4, 1 < 4, aller à gauche (éliminer la moitié droite !)

In [13]:
# Exemple ABR :
#       4
#      / \
#     2   6
#    / \ / \
#   1  3 5  7

bst_root = create_tree_node(4)
bst_root['left'] = create_tree_node(2)
bst_root['right'] = create_tree_node(6)
bst_root['left']['left'] = create_tree_node(1)
bst_root['left']['right'] = create_tree_node(3)
bst_root['right']['left'] = create_tree_node(5)
bst_root['right']['right'] = create_tree_node(7)

# Recherche dans un ABR
def bst_search(root, target):
    if not root or root['val'] == target:
        return root
    
    if target < root['val']:
        return bst_search(root['left'], target)
    else:
        return bst_search(root['right'], target)

# Test
result = bst_search(bst_root, 5)
print(f"Trouvé 5 : {result['val'] if result else None}")
print(f"Trouvé 10 : {bst_search(bst_root, 10)}")

Trouvé 5 : 5
Trouvé 10 : None


---

## Partie 2 : Parcours d'arbres (DFS)

### Parcours en profondeur (DFS)

Trois façons de parcourir - la différence est **quand on visite la racine** :

**Exemple d'arbre :**
```
        1
       / \
      2   3
     / \
    4   5
```

**Préfixe (Préordre) : Racine → Gauche → Droite**
```
Pensez : "Visitez le nœud D'ABORD, puis explorez ses enfants"

Trace d'exécution (détaillée) :
  Commencer à la racine (1) :
    ✓ Visiter 1 → Afficher "1"
    → Aller à gauche vers le nœud 2 :
      ✓ Visiter 2 → Afficher "2"
      → Aller à gauche vers le nœud 4 :
        ✓ Visiter 4 → Afficher "4" (pas d'enfants, retour)
      ← Retour au nœud 2
      → Aller à droite vers le nœud 5 :
        ✓ Visiter 5 → Afficher "5" (pas d'enfants, retour)
      ← Retour au nœud 2 (terminé)
    ← Retour à la racine (1)
    → Aller à droite vers le nœud 3 :
      ✓ Visiter 3 → Afficher "3" (pas d'enfants, retour)
    ← Retour à la racine (1) (terminé)

Résultat : 1, 2, 4, 5, 3
Modèle : Toujours afficher AVANT d'aller vers les enfants
```

**Infixe (Inordre) : Gauche → Racine → Droite**
```
Pensez : "Explorez à gauche D'ABORD, puis visitez le nœud, puis explorez à droite"

Trace d'exécution (détaillée) :
  Commencer à la racine (1) :
    → Aller à gauche vers le nœud 2 (ne pas visiter 1 encore !) :
      → Aller à gauche vers le nœud 4 (ne pas visiter 2 encore !) :
        ✓ Visiter 4 → Afficher "4" (pas d'enfants, retour)
      ← Retour au nœud 2
      ✓ Visiter 2 → Afficher "2" (visiter MAINTENANT, entre gauche et droite !)
      → Aller à droite vers le nœud 5 :
        ✓ Visiter 5 → Afficher "5" (pas d'enfants, retour)
      ← Retour au nœud 2 (terminé)
    ← Retour à la racine (1)
    ✓ Visiter 1 → Afficher "1" (visiter MAINTENANT, après le sous-arbre gauche !)
    → Aller à droite vers le nœud 3 :
      ✓ Visiter 3 → Afficher "3" (pas d'enfants, retour)
    ← Retour à la racine (1) (terminé)

Résultat : 4, 2, 5, 1, 3
Modèle : Toujours afficher APRÈS le sous-arbre gauche, AVANT le sous-arbre droit
Note : Pour un ABR, l'infixe donne l'ordre trié !
```

**Postfixe (Postordre) : Gauche → Droite → Racine**
```
Pensez : "Explorez TOUS les enfants D'ABORD, puis visitez le nœud EN DERNIER"

Trace d'exécution (détaillée) :
  Commencer à la racine (1) :
    → Aller à gauche vers le nœud 2 (ne pas visiter 1 encore !) :
      → Aller à gauche vers le nœud 4 (ne pas visiter 2 encore !) :
        ✓ Visiter 4 → Afficher "4" (pas d'enfants, retour)
      ← Retour au nœud 2
      → Aller à droite vers le nœud 5 (ne pas visiter 2 encore !) :
        ✓ Visiter 5 → Afficher "5" (pas d'enfants, retour)
      ← Retour au nœud 2
      ✓ Visiter 2 → Afficher "2" (visiter MAINTENANT, après LES DEUX enfants !)
    ← Retour à la racine (1)
    → Aller à droite vers le nœud 3 (ne pas visiter 1 encore !) :
      ✓ Visiter 3 → Afficher "3" (pas d'enfants, retour)
    ← Retour à la racine (1)
    ✓ Visiter 1 → Afficher "1" (EN DERNIER ! Après que TOUS les enfants soient explorés !)

Résultat : 4, 5, 2, 3, 1
Modèle : Toujours afficher APRÈS avoir visité tous les enfants
Cas d'utilisation : Bon pour supprimer des arbres (supprimer les enfants avant le parent)
```

**Astuce mnémotechnique :** Pré=avant, In=au milieu, Post=après (quand visiter la racine)

In [14]:
# Parcours préfixe : Racine → Gauche → Droite
# "Pré" = avant les enfants, donc visiter la racine AVANT d'aller vers les enfants
def preorder(root):
    if not root:  # Cas de base : arbre vide, rien à faire
        return
    
    print(root['val'], end=" ")  # Étape 1 : Visiter la racine D'ABORD - utiliser la clé du dictionnaire
    preorder(root['left'])       # Étape 2 : Puis visiter le sous-arbre gauche
    preorder(root['right'])      # Étape 3 : Puis visiter le sous-arbre droit

# Parcours infixe : Gauche → Racine → Droite
# "In" = au milieu, donc visiter la racine ENTRE gauche et droite
def inorder(root):
    if not root:
        return
    
    inorder(root['left'])        # Étape 1 : Visiter le sous-arbre gauche D'ABORD
    print(root['val'], end=" ")  # Étape 2 : Visiter la racine AU MILIEU (entre gauche et droite)
    inorder(root['right'])       # Étape 3 : Puis visiter le sous-arbre droit

# Parcours postfixe : Gauche → Droite → Racine
# "Post" = après les enfants, donc visiter la racine APRÈS avoir visité les enfants
def postorder(root):
    if not root:
        return
    
    postorder(root['left'])      # Étape 1 : Visiter le sous-arbre gauche D'ABORD
    postorder(root['right'])     # Étape 2 : Visiter le sous-arbre droit DEUXIÈMEMENT
    print(root['val'], end=" ")  # Étape 3 : Visiter la racine EN DERNIER (après les deux enfants)

# Test sur l'arbre :    1
#                     / \
#                    2   3
#                   / \
#                  4   5

print("Préfixe :", end=" ")
preorder(root)  # 1 2 4 5 3
print()

print("Infixe :", end=" ")
inorder(root)  # 4 2 5 1 3
print()

print("Postfixe :", end=" ")
postorder(root)  # 4 5 2 3 1
print()

Préfixe : 1 2 4 5 3 
Infixe : 4 2 5 1 3 
Postfixe : 4 5 2 3 1 


In [15]:
# Versions itératives (utilisant des piles)

def preorder_iterative(root):
    """Parcours préfixe utilisant une pile (itératif)"""
    if not root:
        return []
    
    result = []
    stack = [root]
    
    while stack:
        node = stack.pop()
        result.append(node['val'])
        
        # Empiler droite d'abord, puis gauche (pour que gauche soit dépilée en premier)
        if node['right']:
            stack.append(node['right'])
        if node['left']:
            stack.append(node['left'])
    
    return result

def inorder_iterative(root):
    """Parcours infixe utilisant une pile (itératif)"""
    result = []
    stack = []
    current = root
    
    # while stack or current:
    #     # Aller jusqu'au nœud le plus à gauche
    #     while current:
    #         stack.append(current)
    #         current = current['left']
        
    #     # Traiter le nœud
    #     current = stack.pop()
    #     result.append(current['val'])
        
    #     # Passer à droite
    #     current = current['right']

    while stack or current:
    # Aller jusqu'au nœud le plus à gauche
        if current:
            stack.append(current)
            current = current['left']
        else:
        # Traiter le nœud
            current = stack.pop()
            result.append(current['val'])
            
            # Passer à droite
            current = current['right']
    
    return result

print(f"Préfixe itératif : {preorder_iterative(root)}")
print(f"Infixe itératif : {inorder_iterative(root)}")

Préfixe itératif : [1, 2, 4, 5, 3]
Infixe itératif : [4, 2, 5, 1, 3]


In [16]:
# 状态机模型S ={stack,current}
def inorder_fsm(root):
    # --- state ---
    state = {
        "current": root,
        "stack": [],
        "result": []
    }

    def push(node):
        state["stack"].append(node)

    def pop():
        return state["stack"].pop()

    def visit(node):
        state["result"].append(node["val"])

    def step():
        """
        One transition step.
        Keeps your rule shape unchanged:
          if current: push; go left
          else: pop; visit; go right
        Returns False when halted.
        """
        current = state["current"]
        stack = state["stack"]

        # halt condition
        if current is None and not stack:
            return False

        # ---- your rule (unchanged form) ----
        if current is not None:
            push(current)
            state["current"] = current["left"]
        else:
            node = pop()
            visit(node)
            state["current"] = node["right"]
        # -----------------------------------

        return True

    while step():
        pass

    return state["result"]

---

## Partie 3 : Parcours en largeur (BFS)

### Parcours par niveau

**BFS** visite les nœuds niveau par niveau, de gauche à droite.

**Utilisations :**
- Trouver le plus court chemin (graphes non pondérés)
- Affichage par niveau
- Trouver les nœuds à une profondeur spécifique

**Implémentation :** Utiliser une file d'attente !

In [17]:
from collections import deque

def level_order(root):
    """
    BFS : Visiter les nœuds niveau par niveau
    """
    if not root:
        return []
    
    result = []
    queue = deque([root])
    
    while queue:
        node = queue.popleft()
        result.append(node['val'])
        
        if node['left']:
            queue.append(node['left'])
        if node['right']:
            queue.append(node['right'])
    
    return result

# Test sur l'arbre :    1
#                     / \
#                    2   3
#                   / \
#                  4   5

print(f"Parcours par niveau : {level_order(root)}")  # [1, 2, 3, 4, 5]

Parcours par niveau : [1, 2, 3, 4, 5]


In [18]:
# Parcours par niveau avec séparation des niveaux
def level_order_by_level(root):
    """
    Retourne une liste de listes, chaque liste représente un niveau
    """
    if not root:
        return []
    
    result = []
    queue = deque([root])
    
    while queue:
        level_size = len(queue)
        level = []
        
        for _ in range(level_size):
            node = queue.popleft()
            level.append(node['val'])
            
            if node['left']:
                queue.append(node['left'])
            if node['right']:
                queue.append(node['right'])
        
        result.append(level)
    
    return result

print(f"Parcours par niveau (séparé) : {level_order_by_level(root)}")
# Sortie : [[1], [2, 3], [4, 5]]

Parcours par niveau (séparé) : [[1], [2, 3], [4, 5]]


---

## Partie 4 : Problèmes courants sur les arbres

In [19]:
# Problème 1 : Profondeur maximale (Hauteur)
def max_depth(root):
    """Trouver la profondeur maximale de l'arbre"""
    if not root:
        return 0
    
    left_depth = max_depth(root['left'])
    right_depth = max_depth(root['right'])
    
    return 1 + max(left_depth, right_depth)

print(f"Profondeur max : {max_depth(root)}")

# Problème 2 : Vérifier si deux arbres sont identiques
def is_same_tree(p, q):
    """Vérifier si deux arbres sont identiques"""
    if not p and not q:
        return True
    if not p or not q:
        return False
    
    return (p['val'] == q['val'] and
            is_same_tree(p['left'], q['left']) and
            is_same_tree(p['right'], q['right']))

# Problème 3 : Inverser un arbre binaire
def invert_tree(root):
    """Inverser (miroir) un arbre binaire"""
    if not root:
        return None
    
    # Échanger les enfants
    root['left'], root['right'] = root['right'], root['left']
    
    # Inverser récursivement les sous-arbres
    invert_tree(root['left'])
    invert_tree(root['right'])
    
    return root

Profondeur max : 3


---

## Partie 5 : Problèmes d'entraînement

### Problème 1 : Valider un arbre binaire de recherche
Vérifier si un arbre binaire est un ABR valide.

**Indice :** Utiliser le parcours infixe ou vérifier les bornes

In [ ]:
# Votre solution ici :
def is_valid_bst(root):
    # TODO : Implémenter
    # all left descents < node.val < all right descents
    def validate(node, low=float('-inf'), high=float('inf')):
        if not node:
            return True
        if not (low < node["val"] < high):
            return False
        
        return validate(node["left"],low,node["val"]) and validate(node["right"],node["val"],high)
        
    
    return validate(root)
       

### Problème 2 : Arbre symétrique
Vérifier si un arbre est symétrique (miroir de lui-même).

**Exemple :**
```
    1
   / \
  2   2
 / \ / \
3  4 4  3
```
C'est symétrique !

In [21]:
# Votre solution ici :
def is_symmetric(root):
    # TODO : Implémenter
    def is_mirror(t1,t2):
        if not t1 and not t2:
            return True
        if not t1 or not t2:
            return False
        
        return (t1["val"] == t2["val"] and
                is_mirror(t1["left"], t2["right"]) and
                is_mirror(t1["right"], t2["left"]))
   
    if not root:
        return True
    return is_mirror(root["left"], root["right"])

### Problème 3 : Somme de chemin
Vérifier s'il existe un chemin racine-feuille avec une somme donnée.

**Exemple :** Arbre `[5,4,8,11,null,13,4,7,2,null,null,null,1]`, somme=22 → True
(Chemin : 5→4→11→2)

In [22]:
# Votre solution ici :
def has_path_sum(root, target_sum):
    # TODO : Implémenter
    if not root:
        return False
    remain = target_sum - root["val"]

    if not root["left"] and not root["right"]:
        return remain == 0
    
    return has_path_sum(root["left"],target_sum - root["val"]) or has_path_sum(root["right"],target_sum - root["val"])
   
    


---

## Solutions (Essayez d'abord !)

<details>
<summary>Cliquez pour révéler les solutions</summary>

### Solution 1 : Valider un ABR
```python
def is_valid_bst(root):
    """Valider un ABR avec des nœuds dictionnaires"""
    def validate(node, min_val, max_val):
        if not node:
            return True
        
        if node['val'] <= min_val or node['val'] >= max_val:
            return False
        
        return (validate(node['left'], min_val, node['val']) and
                validate(node['right'], node['val'], max_val))
    
    return validate(root, float('-inf'), float('inf'))
```

### Solution 2 : Arbre symétrique
```python
def is_symmetric(root):
    """Vérifier si l'arbre est symétrique avec des nœuds dictionnaires"""
    def is_mirror(left, right):
        if not left and not right:
            return True
        if not left or not right:
            return False
        
        return (left['val'] == right['val'] and
                is_mirror(left['left'], right['right']) and
                is_mirror(left['right'], right['left']))
    
    if not root:
        return True
    return is_mirror(root['left'], root['right'])
```

### Solution 3 : Somme de chemin
```python
def has_path_sum(root, target_sum):
    """Vérifier si une somme de chemin existe avec des nœuds dictionnaires"""
    if not root:
        return False
    
    if not root['left'] and not root['right']:
        return root['val'] == target_sum
    
    remaining = target_sum - root['val']
    return (has_path_sum(root['left'], remaining) or
            has_path_sum(root['right'], remaining))
```

</details>

---

## Points clés à retenir

✅ **Structure des arbres :**
- Hiérarchique avec racine, nœuds, arêtes
- Arbre binaire : max 2 enfants par nœud
- ABR : gauche < nœud < droite

✅ **Parcours DFS :**
- Préfixe : Racine → Gauche → Droite
- Infixe : Gauche → Racine → Droite (donne l'ordre trié pour un ABR)
- Postfixe : Gauche → Droite → Racine
- Peut être récursif ou itératif (avec pile)

✅ **BFS (Parcours par niveau) :**
- Visiter niveau par niveau
- Utiliser une file d'attente
- Bon pour les problèmes de plus court chemin

✅ **Quand utiliser :**
- DFS : Quand vous devez explorer en profondeur (problèmes de chemin, propriétés d'arbre)
- BFS : Quand vous avez besoin niveau par niveau (plus court chemin, problèmes de niveau)

---

## Devoirs

1. Compléter tous les problèmes d'entraînement
2. Résoudre les LeetCode Facile :
   - [104. Maximum Depth of Binary Tree](https://leetcode.com/problems/maximum-depth-of-binary-tree/)
   - [226. Invert Binary Tree](https://leetcode.com/problems/invert-binary-tree/)
   - [101. Symmetric Tree](https://leetcode.com/problems/symmetric-tree/)
3. S'entraîner à dessiner des arbres et tracer les parcours manuellement

**Prochaine session :** Semaine 6, Session 1 - Tas & Bases des graphes